# Hybrid legal citation extractor

Turns a legal document into a JSON list of citations, for use in a hallucination eval.

**Pipeline**

1. **Detect (eyecite).** Deterministic parsing of case cites, statutes, regs, short forms, *Id.* and *supra*, with exact character offsets.
2. **Enrich (Claude).** For each detected cite, Claude adds the proposition it is cited for, any quoted language, the signal and the pin cite. Claude also lists citations eyecite missed (rules, Restatements, odd formatting).
3. **Validate and reconcile (code).** Nothing from the model is trusted blindly: every citation, quote and supporting sentence Claude returns must appear verbatim in the document, or it is flagged. Disagreements are marked `needs_review`.
4. **Score the extractor** against a hand-labeled gold list, so you know its precision and recall before using it inside the eval.
5. **(Optional) Verify existence** with CourtListener's citation-lookup API. This is the step that actually catches hallucinated cases.

The notebook runs without an API key: stage 2 is skipped and you get the eyecite-only output. Set `ANTHROPIC_API_KEY` (or log in with `ant auth login`) to run the full hybrid.

In [1]:
%pip install -q eyecite anthropic pydantic requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, logging, os, re
from dataclasses import dataclass, field, asdict
from typing import Literal, Optional

from eyecite import get_citations, resolve_citations
from eyecite.models import (
    FullCaseCitation, FullLawCitation, FullJournalCitation, ShortCaseCitation,
    IdCitation, SupraCitation, UnknownCitation,
)

logging.getLogger("eyecite").setLevel(logging.ERROR)  # eyecite logs noisy "overlap" warnings

MODEL = "claude-opus-5-5"

## Sample document

A short fake brief with real cites, a statute, a regulation, short forms, a Federal Rule and a Restatement (which eyecite misses), and two cites that should fail verification: *Varghese*, one of the fabricated cases from *Mata v. Avianca*, and an invented *Martinez* cite. Replace `DOCUMENT` with your own text, or load a file.

In [3]:
DOCUMENT = """MEMORANDUM IN SUPPORT OF MOTION TO DISMISS

Defendant moves to dismiss under Fed. R. Civ. P. 12(b)(6). A complaint must plead "enough facts to state a claim to relief that is plausible on its face." Bell Atlantic Corp. v. Twombly, 550 U.S. 544, 570 (2007). Threadbare recitals do not suffice. Ashcroft v. Iqbal, 556 U.S. 662, 678 (2009). Id. at 679.

Plaintiff relies on Varghese v. China Southern Airlines Co., 925 F.3d 1339 (11th Cir. 2019), for the proposition that the limitations period is tolled during bankruptcy. But see Mata v. Avianca, Inc., 678 F. Supp. 3d 443 (S.D.N.Y. 2023) (sanctioning counsel for citing nonexistent cases). Plaintiff also cites Martinez v. Uber Techs., Inc., 612 F.4th 88 (2d Cir. 2024).

Under 42 U.S.C. § 1983, a plaintiff must allege a deprivation under color of state law. See also 28 C.F.R. § 35.130(b)(7). Strict liability for defective products follows Restatement (Second) of Torts § 402A (1965). Cal. Civ. Code § 1714(a) separately imposes a general duty of care.

As the Court explained in Twombly, 550 U.S. at 556, plausibility is not probability. See Iqbal, supra, at 680.
"""
print(len(DOCUMENT), "characters")

1118 characters


## Stage 1: detect with eyecite

Each detection becomes a `Citation` record with exact offsets into the document.

One patch is applied. In eyecite 2.7.x, a full case cite's `year` can be taken from a *later* citation's parenthetical when several cites share a paragraph (for example, *Twombly* reported as 2009). `fix_year` re-reads the year from the parenthetical that directly follows the cite and falls back to `None`.

In [4]:
@dataclass
class Citation:
    id: str
    raw_text: str
    start: int
    end: int
    type: str                      # case | short_case | id | supra | statute | regulation | rule | secondary | journal | other
    normalized: dict = field(default_factory=dict)
    antecedent_id: Optional[str] = None   # for Id./supra/short forms
    proposition: Optional[str] = None
    supporting_sentence: Optional[str] = None
    quoted_text: Optional[str] = None
    signal: Optional[str] = None
    pin_cite: Optional[str] = None
    source: str = "eyecite"        # eyecite | llm | both
    flags: list = field(default_factory=list)

    @property
    def needs_review(self):
        return bool(self.flags)


YEAR_AFTER = re.compile(r"^[^()]{0,80}?\(([^()]*?)\b(1[6-9]\d\d|20\d\d)\)")

def fix_year(text, end):
    m = YEAR_AFTER.match(text[end:end + 200])
    return (m.group(2), m.group(1).strip() or None) if m else (None, None)


REG_REPORTERS = ("C.F.R.", "Fed. Reg.")

def classify(c):
    if isinstance(c, FullCaseCitation): return "case"
    if isinstance(c, ShortCaseCitation): return "short_case"
    if isinstance(c, IdCitation): return "id"
    if isinstance(c, SupraCitation): return "supra"
    if isinstance(c, FullJournalCitation): return "journal"
    if isinstance(c, FullLawCitation):
        return "regulation" if c.groups.get("reporter") in REG_REPORTERS else "statute"
    return "other"


def detect(text):
    found = [c for c in get_citations(text) if not isinstance(c, UnknownCitation)]
    records, by_obj = [], {}
    for i, c in enumerate(found):
        start, end = c.span()
        kind = classify(c)
        md_ = c.metadata
        norm = {k: v for k, v in c.groups.items() if v}
        for key in ("plaintiff", "defendant", "court", "pin_cite", "parenthetical"):
            v = getattr(md_, key, None)
            if v: norm[key] = v
        if kind == "case":
            year, court_paren = fix_year(text, end)
            norm["year"] = year
            if court_paren: norm["court_parenthetical"] = court_paren
        rec = Citation(id=f"c{i+1}", raw_text=text[start:end], start=start, end=end,
                       type=kind, normalized=norm, pin_cite=getattr(md_, "pin_cite", None))
        records.append(rec)
        by_obj[id(c)] = rec

    # Link short forms / Id. / supra to the full cite they refer to.
    for resource, cites in resolve_citations(found).items():
        full = next((by_obj[id(x)] for x in cites if by_obj.get(id(x)) and by_obj[id(x)].type in ("case", "statute", "regulation", "journal")), None)
        for x in cites:
            rec = by_obj.get(id(x))
            if rec and full and rec is not full:
                rec.antecedent_id = full.id
    for rec in records:
        if rec.type in ("id", "supra", "short_case") and not rec.antecedent_id:
            rec.flags.append("unresolved_short_form")
    return records


detected = detect(DOCUMENT)
for r in detected:
    print(f"{r.id:4} {r.type:11} {r.raw_text!r:40} -> {r.antecedent_id or ''} {r.normalized.get('year') or ''}")

c1   case        '550 U.S. 544'                           ->  2007
c2   case        '556 U.S. 662'                           ->  2009
c3   id          'Id. at 679'                             -> c2 
c4   case        '925 F.3d 1339'                          ->  2019
c5   case        '678 F. Supp. 3d 443'                    ->  2023
c6   case        '612 F.4th 88'                           ->  2024
c7   statute     '42 U.S.C. § 1983'                       ->  
c8   regulation  '28 C.F.R. § 35.130'                     ->  
c9   statute     'Cal. Civ. Code § 1714'                  ->  
c10  short_case  '550 U.S. at 556'                        -> c1 
c11  supra       'supra, at 680'                          -> c2 


## Stage 2: enrich with Claude

Claude gets the full document plus the numbered eyecite detections and returns structured output (a Pydantic schema, so the JSON always parses). It is told to copy text **verbatim** so stage 3 can check every string against the document.

The request uses server-side fallbacks (`fallbacks="default"`): if a safety classifier declines the request, the API retries it on a fallback model inside the same call. Remove those two arguments if you don't want that.

For very long documents, split on section boundaries and run this per chunk, passing each chunk's detections with offsets relative to the chunk.

In [5]:
from pydantic import BaseModel, Field

Signal = Literal["none", "see", "see also", "cf.", "but see", "but cf.", "see generally", "compare", "e.g.", "accord", "contra", "other"]

class Enrichment(BaseModel):
    id: str = Field(description="The id of the detected citation, e.g. c3")
    proposition: str = Field(description="One sentence: the legal proposition the document cites this authority for, in your own words")
    supporting_sentence: str = Field(description="The sentence from the document that makes the claim, copied verbatim")
    quoted_text: Optional[str] = Field(description="Language the document puts in quotation marks and attributes to this authority, copied verbatim without the quote marks; null if none")
    signal: Signal
    pin_cite: Optional[str] = Field(description="Pin cite as written, e.g. '570' or 'at 679'; null if none")
    looks_malformed: bool = Field(description="True if the citation's format looks wrong (bad reporter abbreviation, impossible volume, mismatched court/year)")

class MissedCitation(BaseModel):
    raw_text: str = Field(description="The citation exactly as it appears in the document, verbatim")
    type: Literal["case", "statute", "regulation", "rule", "secondary", "constitution", "other"]
    proposition: str
    supporting_sentence: str = Field(description="Verbatim sentence from the document")

class ExtractionResult(BaseModel):
    enrichments: list[Enrichment]
    missed: list[MissedCitation] = Field(description="Citations to legal authority present in the document but absent from the detected list")


SYSTEM = """You annotate citations in legal documents for a citation-accuracy audit.
You never correct, complete, or normalize a citation: report what the document says, exactly as written, even if you believe it is wrong or fabricated.
Every field described as verbatim must be copied character-for-character from the document so it can be string-matched."""


def build_prompt(text, detected):
    listing = "\n".join(f"{r.id}\t[{r.start}:{r.end}]\t{r.type}\t{r.raw_text}" for r in detected)
    return f"""<document>
{text}
</document>

<detected_citations>
id\tspan\ttype\traw_text
{listing}
</detected_citations>

For every detected citation, return an enrichment. For Id., supra and short forms, describe the proposition they are cited for at that location.
Then list any citations to legal authority the detector missed (court rules, Restatements, constitutional provisions, oddly formatted cites). Do not re-list detected ones."""


def enrich(text, detected):
    import anthropic
    client = anthropic.Anthropic()
    response = client.beta.messages.parse(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM,
        messages=[{"role": "user", "content": build_prompt(text, detected)}],
        output_format=ExtractionResult,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    if response.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {response.stop_details}")
    return response.parsed_output


def have_credentials():
    if os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"):
        return True
    import anthropic
    client = anthropic.Anthropic()   # also picks up an `ant auth login` profile
    return bool(client.api_key or client.auth_token or getattr(client, "credentials", None))


USE_LLM = have_credentials()
llm_result = enrich(DOCUMENT, detected) if USE_LLM else None
print("LLM stage ran" if llm_result else "No Anthropic credentials found: skipping stage 2 (eyecite-only output)")

No Anthropic credentials found: skipping stage 2 (eyecite-only output)


## Stage 3: validate and reconcile

This is where the extractor protects itself from its own hallucinations:

- A **missed** cite from Claude is kept only if its `raw_text` is found verbatim in the document; otherwise it is dropped and logged in `rejected`. Found ones get real offsets.
- `supporting_sentence` and `quoted_text` must appear verbatim, or the cite gets a `*_not_in_document` flag.
- Enrichments that reference unknown ids are rejected; detections Claude skipped are flagged `not_enriched`.
- Claude's `looks_malformed` becomes a flag. Stage 5 decides whether the case exists.

In [6]:
def norm_ws(s):
    return re.sub(r"\s+", " ", s).strip()

def in_doc(snippet, text):
    return bool(snippet) and norm_ws(snippet) in norm_ws(text)

def find_span(snippet, text, taken):
    """First exact occurrence not overlapping an existing citation."""
    for m in re.finditer(re.escape(snippet), text):
        if not any(m.start() < e and s < m.end() for s, e in taken):
            return m.start(), m.end()
    return None


def reconcile(text, detected, llm_result):
    cites = {r.id: r for r in detected}
    rejected = []
    if llm_result is None:
        return list(cites.values()), rejected

    seen = set()
    for e in llm_result.enrichments:
        rec = cites.get(e.id)
        if rec is None:
            rejected.append({"reason": "unknown_id", "item": e.model_dump()})
            continue
        seen.add(e.id)
        rec.source = "both"
        rec.proposition = e.proposition
        rec.supporting_sentence = e.supporting_sentence
        rec.quoted_text = e.quoted_text
        rec.signal = e.signal
        if e.pin_cite and rec.pin_cite and norm_ws(e.pin_cite).removeprefix("at ") != norm_ws(rec.pin_cite).removeprefix("at "):
            rec.flags.append(f"pin_cite_disagreement: eyecite={rec.pin_cite!r} llm={e.pin_cite!r}")
        rec.pin_cite = rec.pin_cite or e.pin_cite
        if not in_doc(e.supporting_sentence, text):
            rec.flags.append("supporting_sentence_not_in_document")
        if e.quoted_text and not in_doc(e.quoted_text, text):
            rec.flags.append("quoted_text_not_in_document")
        if e.looks_malformed:
            rec.flags.append("llm_says_malformed")

    for rec in cites.values():
        if rec.id not in seen:
            rec.flags.append("not_enriched")

    taken = [(r.start, r.end) for r in cites.values()]
    n = len(cites)
    for m in llm_result.missed:
        span = find_span(m.raw_text, text, taken)
        if span is None:
            rejected.append({"reason": "missed_cite_not_in_document", "item": m.model_dump()})
            continue
        n += 1
        rec = Citation(id=f"c{n}", raw_text=m.raw_text, start=span[0], end=span[1], type=m.type,
                       proposition=m.proposition, supporting_sentence=m.supporting_sentence, source="llm")
        rec.flags.append("llm_only_detection")
        if not in_doc(m.supporting_sentence, text):
            rec.flags.append("supporting_sentence_not_in_document")
        cites[rec.id] = rec
        taken.append(span)

    return sorted(cites.values(), key=lambda r: r.start), rejected


citations, rejected = reconcile(DOCUMENT, detected, llm_result)
print(f"{len(citations)} citations, {sum(c.needs_review for c in citations)} need review, {len(rejected)} rejected model outputs")

11 citations, 0 need review, 0 rejected model outputs


## Output JSON

In [7]:
def to_json(citations, rejected):
    return {
        "model": MODEL if llm_result else None,
        "citations": [{**asdict(c), "needs_review": c.needs_review} for c in citations],
        "rejected_model_outputs": rejected,
    }

output = to_json(citations, rejected)
with open("citations.json", "w") as f:
    json.dump(output, f, indent=2)

print(json.dumps(output["citations"][:3], indent=2))
print("... wrote citations.json")

[
  {
    "id": "c1",
    "raw_text": "550 U.S. 544",
    "start": 231,
    "end": 243,
    "type": "case",
    "normalized": {
      "volume": "550",
      "reporter": "U.S.",
      "page": "544",
      "plaintiff": "Bell Atlantic Corp.",
      "defendant": "Twombly",
      "court": "scotus",
      "pin_cite": "570",
      "year": "2007"
    },
    "antecedent_id": null,
    "proposition": null,
    "supporting_sentence": null,
    "quoted_text": null,
    "signal": null,
    "pin_cite": "570",
    "source": "eyecite",
    "flags": [],
    "needs_review": false
  },
  {
    "id": "c2",
    "raw_text": "556 U.S. 662",
    "start": 312,
    "end": 324,
    "type": "case",
    "normalized": {
      "volume": "556",
      "reporter": "U.S.",
      "page": "662",
      "plaintiff": "Ashcroft",
      "defendant": "Iqbal",
      "court": "scotus",
      "pin_cite": "678",
      "year": "2009"
    },
    "antecedent_id": null,
    "proposition": null,
    "supporting_sentence": null,
    "quo

## Stage 4: score the extractor against a gold list

Hand-label the citations in a few real documents (just the raw text is enough to start). A detection counts as correct when its span overlaps a gold cite's span. Grow this to 20 to 50 documents that look like what your eval will see before trusting the numbers.

In [8]:
GOLD = [
    "Fed. R. Civ. P. 12(b)(6)",
    "550 U.S. 544",
    "556 U.S. 662",
    "Id. at 679",
    "925 F.3d 1339",
    "678 F. Supp. 3d 443",
    "612 F.4th 88",
    "42 U.S.C. § 1983",
    "28 C.F.R. § 35.130",
    "Restatement (Second) of Torts § 402A",
    "Cal. Civ. Code § 1714",
    "550 U.S. at 556",
    "supra, at 680",
]

def gold_spans(text, gold):
    spans = []
    for g in gold:
        i = text.find(g)
        assert i >= 0, f"gold cite not in document: {g}"
        spans.append((i, i + len(g), g))
    return spans

def score(citations, text, gold):
    gs = gold_spans(text, gold)
    overlaps = lambda c, g: c.start < g[1] and g[0] < c.end
    tp_pred = [c for c in citations if any(overlaps(c, g) for g in gs)]
    hit_gold = [g for g in gs if any(overlaps(c, g) for c in citations)]
    precision = len(tp_pred) / len(citations) if citations else 0.0
    recall = len(hit_gold) / len(gs)
    return {
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "missed": [g[2] for g in gs if g not in hit_gold],
        "false_positives": [c.raw_text for c in citations if c not in tp_pred],
    }

print("eyecite only:", json.dumps(score(detect(DOCUMENT), DOCUMENT, GOLD), indent=2))
if llm_result:
    print("hybrid:      ", json.dumps(score(citations, DOCUMENT, GOLD), indent=2))

eyecite only: {
  "precision": 1.0,
  "recall": 0.846,
  "missed": [
    "Fed. R. Civ. P. 12(b)(6)",
    "Restatement (Second) of Torts \u00a7 402A"
  ],
  "false_positives": []
}


## Stage 5 (optional): does the cited case exist?

CourtListener's citation-lookup API takes text and reports, for each reporter citation, whether it matches a real opinion (`status` 200) or not (404). Get a free token at courtlistener.com and set `COURTLISTENER_TOKEN`. This checks existence only; checking that the case supports the stated `proposition` is the next stage of the eval.

In [9]:
import requests

def courtlistener_lookup(text, token):
    r = requests.post(
        "https://www.courtlistener.com/api/rest/v4/citation-lookup/",
        headers={"Authorization": f"Token {token}"},
        data={"text": text},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

token = os.environ.get("COURTLISTENER_TOKEN")
if token:
    for hit in courtlistener_lookup(DOCUMENT, token):
        names = [c.get("case_name") for c in hit.get("clusters", [])]
        print(hit["status"], hit["citation"], names or "NOT FOUND")
else:
    print("Set COURTLISTENER_TOKEN to run existence checks.")

Set COURTLISTENER_TOKEN to run existence checks.


## Next steps

- Build the gold set from real briefs and AI-generated answers, and track precision and recall per citation type.
- Feed `citations.json` into the verification stage: existence (stage 5), then whether the source actually supports `proposition` and contains `quoted_text`.
- Pin the eyecite version, and re-check the year patch when upgrading.